# NpuKit — 8×8 int8 matmul on PYNQ-Z2

Downloads `npukit.bit`, runs the systolic array over AXI-Lite, and checks **C** against NumPy.

Requires `npukit.bit` in this folder (or set `BIT_PATH` below).

In [ ]:
import struct
import time

import numpy as np
from pynq import Bitstream, MMIO

N = 8
BASE = 0x43C0_0000
SPAN = 0x1000

REG_ID = 0x000
REG_VERSION = 0x004
REG_STATUS = 0x008
REG_CTRL = 0x00C
REG_N = 0x010
OFF_A = 0x100
OFF_B = 0x200
OFF_C = 0x400

ID_MAGIC = 0x4E50554B
BIT_PATH = "/home/xilinx/jupyter_notebooks/npukit.bit"

In [ ]:
def pack_i8(mat):
    flat = np.asarray(mat, dtype=np.int8).reshape(-1)
    words = []
    for i in range(0, flat.size, 4):
        words.append(struct.unpack("<I", flat[i : i + 4].tobytes())[0])
    return words


def write_matrix(mmio, offset, mat):
    for i, w in enumerate(pack_i8(mat)):
        mmio.write(offset + 4 * i, int(w))


def read_c(mmio):
    out = np.zeros((N, N), dtype=np.int32)
    for i in range(N * N):
        out.flat[i] = np.int32(mmio.read(OFF_C + 4 * i))
    return out

## Download bitstream

In [ ]:
print("Downloading", BIT_PATH)
Bitstream(BIT_PATH).download()

mmio = MMIO(BASE, SPAN)
ident = mmio.read(REG_ID)
assert ident == ID_MAGIC, f"BAD ID: 0x{ident:08X} (expected 0x{ID_MAGIC:08X})"
print(f"ID OK  version=0x{mmio.read(REG_VERSION):08X}  N={mmio.read(REG_N)}")

## Build A, B and golden C (NumPy)

In [ ]:
A = np.arange(1, N + 1, dtype=np.int8).reshape(N, 1) * np.ones((1, N), dtype=np.int8)
B = np.ones((N, N), dtype=np.int8)
C_ref = A.astype(np.int32) @ B.astype(np.int32)
A, B, C_ref

## Run on NpuKit and check result

In [ ]:
write_matrix(mmio, OFF_A, A)
write_matrix(mmio, OFF_B, B)

mmio.write(REG_CTRL, 0x2)  # CLEAR
mmio.write(REG_CTRL, 0x1)  # START

t0 = time.time()
for _ in range(10000):
    st = mmio.read(REG_STATUS)
    if st & 0x2:
        break
    time.sleep(0.0001)
else:
    raise TimeoutError(f"status=0x{mmio.read(REG_STATUS):X}")

C = read_c(mmio)
dt = time.time() - t0
print(f"done in {dt * 1e3:.2f} ms  status=0x{st:X}")

assert np.array_equal(C, C_ref), (C, C_ref)
print("PASS: C matches NumPy int32 matmul")
C